In [56]:
import csv
import matplotlib.pyplot as plt
import numpy as np

In [57]:
ra_loc = []
dec_loc = []
dVTEC_ra = []
dVTEC_dec = []
with open('/Users/ruby/Downloads/Sample_dVTEC_interpolated_frame (1).csv', mode='r') as csv_file:
    reader = csv.reader(csv_file)
    header = next(reader)  # Optional: skip the header row
    for row in reader:
        ra_loc.append(float(row[1]))
        dec_loc.append(float(row[2]))
        dVTEC_ra.append(float(row[3]))
        dVTEC_dec.append(float(row[4]))

In [58]:
ras_unique = np.unique(ra_loc)
decs_unique = np.unique(dec_loc)
dVTEC_ra_grid = np.full((len(ras_unique), len(decs_unique)), np.nan)
dVTEC_dec_grid = np.full((len(ras_unique), len(decs_unique)), np.nan)
for ra_ind, ra in enumerate(ras_unique):
    for dec_ind, dec in enumerate(decs_unique):
        ind = np.where((ra_loc == ra) & (dec_loc == dec))[0][0]
        dVTEC_ra_grid[ra_ind, dec_ind] = dVTEC_ra[ind]
        dVTEC_dec_grid[ra_ind, dec_ind] = dVTEC_dec[ind]


In [66]:
# Convert from units of TEC/m to TEC/deg
ionospheric_height_m = 500
deg_per_m = np.arcsin(1/ionospheric_height_m)
dVTEC_ra_per_deg = dVTEC_ra_grid / deg_per_m
dVTEC_dec_per_deg = dVTEC_dec_grid / deg_per_m

In [69]:
TEC_integrated = np.full((len(ras_unique), len(decs_unique)), 0, dtype=float)
for ra_ind, ra in enumerate(ras_unique[:-1]):
    for dec_ind, dec in enumerate(decs_unique[:-1]):
        TEC_integrated[ra_ind+1, dec_ind] = TEC_integrated[ra_ind, dec_ind] + dVTEC_ra_per_deg[ra_ind, dec_ind] * (ras_unique[ra_ind+1] - ra)
        TEC_integrated[ra_ind, dec_ind+1] = TEC_integrated[ra_ind, dec_ind] + dVTEC_dec_per_deg[ra_ind, dec_ind] * (decs_unique[ra_ind+1] - dec)

# Make sure it is mean-zero:
TEC_integrated -= np.mean(TEC_integrated)